In [87]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [88]:
df = pd.read_csv("/kaggle/input/datasets/yakupie/asdsadqwewqe/log2.csv")

df.head(5)

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Action,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received
0,57222,53,54587,53,allow,177,94,83,2,30,1,1
1,56258,3389,56258,3389,allow,4768,1600,3168,19,17,10,9
2,6881,50321,43265,50321,allow,238,118,120,2,1199,1,1
3,50553,3389,50553,3389,allow,3327,1438,1889,15,17,8,7
4,50002,443,45848,443,allow,25358,6778,18580,31,16,13,18


In [89]:
# veri isimlerini düzenleme
df = df.rename(columns={
    'Source Port': 'Kaynak Port',
    'Destination Port': 'Hedef Port',
    'NAT Source Port': 'NAT Kaynak Port',
    'NAT Destination Port': 'Nat Hedef Port',
    'Action': 'Aksiyon',
    'Bytes': 'Byte',
    'Bytes Sent': 'Gönderilen Byte',
    'Bytes Received': 'Alınan Byte',
    'Packets': 'Paketler',
    'Elapsed Time (sec)': 'Geçen Süre',
    'pkts_sent': 'Gönderilen Paket',
    'pkts_received': 'Alınan Paket'
})

df.head(5)

,Kaynak Port,Hedef Port,NAT Kaynak Port,Nat Hedef Port,Aksiyon,Byte,Gönderilen Byte,Alınan Byte,Paketler,Geçen Süre,Gönderilen Paket,Alınan Paket
0,57222,53,54587,53,allow,177,94,83,2,30,1,1
1,56258,3389,56258,3389,allow,4768,1600,3168,19,17,10,9
2,6881,50321,43265,50321,allow,238,118,120,2,1199,1,1
3,50553,3389,50553,3389,allow,3327,1438,1889,15,17,8,7
4,50002,443,45848,443,allow,25358,6778,18580,31,16,13,18


In [90]:
# veri hakkinda bilgi almak için
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65532 entries, 0 to 65531
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Kaynak Port       65532 non-null  int64 
 1   Hedef Port        65532 non-null  int64 
 2   NAT Kaynak Port   65532 non-null  int64 
 3   Nat Hedef Port    65532 non-null  int64 
 4   Aksiyon           65532 non-null  object
 5   Byte              65532 non-null  int64 
 6   Gönderilen Byte   65532 non-null  int64 
 7   Alınan Byte       65532 non-null  int64 
 8   Paketler          65532 non-null  int64 
 9   Geçen Süre        65532 non-null  int64 
 10  Gönderilen Paket  65532 non-null  int64 
 11  Alınan Paket      65532 non-null  int64 
dtypes: int64(11), object(1)
memory usage: 6.0+ MB
None


In [91]:
# aksiyonun dağılımına bakıyoruz
print(df['Aksiyon'].value_counts())

Aksiyon
allow         37640
deny          14987
drop          12851
reset-both       54
Name: count, dtype: int64


In [92]:

# 'Aksiyon değerinde ki kategorik verileri 0 ve 1 lere çeviriyoruz ki model anlasın
df['Saldırı'] = (df['Aksiyon'] != 'allow').astype(int)

# yeni column dağılımları
print(df['Saldırı'].value_counts())

Saldırı
0    37640
1    27892
Name: count, dtype: int64


In [93]:
# aksiyon ve saldırı verilerini modelde eğitilirken saklayıp y değerini tahmin etmeye çalışıyoruz
X = df.drop(columns=['Aksiyon', 'Saldırı'])
y = df['Saldırı']

# veriyi %80 eğiti, %20 test olarak bölüyoruz
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [94]:
# modeli tanımlama
model = RandomForestClassifier(n_estimators=100, random_state=42)

# modeli sadece eğitim verileriyle eğitiyoruz
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# doğruluk oranları
print("Doğruluk Skoru : ", accuracy_score(y_test, y_pred))
print("\n--- Performans Raporu ---")
print(classification_report(y_test, y_pred))

Doğruluk Skoru :  0.9996948195620661

--- Performans Raporu ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7528
           1       1.00      1.00      1.00      5579

    accuracy                           1.00     13107
   macro avg       1.00      1.00      1.00     13107
weighted avg       1.00      1.00      1.00     13107

